In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.interpolate import interp1d
import seaborn as sns
import os
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pickle
import datetime
import warnings
import sys
from multiprocessing import Pool, cpu_count

warnings.simplefilter(action='ignore', category=FutureWarning)

from ClassFunctions import precip_time_series, rainfall_analysis

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

BASE_DIR   = '/scratch/hydro4/users/kv25483/MetricEvaluation/Data/'
DIRECTORY  = 'DanishRainData_SVK'
TEMP_RES   = 5
THRESHOLD  = '11h'
N_WORKERS  = 4  # set to cpu_count() to use all available cores

INPUT_DIR   = os.path.join(BASE_DIR, DIRECTORY)
OUTPUT_DIR  = os.path.join(BASE_DIR, 'DanishRainData_Outputs', f'{TEMP_RES}mins_new')
PICKLE_DIR  = os.path.join(BASE_DIR, 'DanishRainDataPickles')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PICKLE_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# Per-file processing function
# ---------------------------------------------------------------------------

def process_file(file):
    """
    Process a single gauge file and save outputs.
    Returns a status string for reporting.
    """
    input_path  = os.path.join(INPUT_DIR,  file)
    output_path = os.path.join(OUTPUT_DIR, f'All_events_{file}')
    pickle_path = os.path.join(PICKLE_DIR, f'{file}.pkl')

    # Skip if already processed
    if os.path.exists(output_path):
        return file, 'skipped'

    try:
        if pd.read_csv(input_path).empty:
            return file, 'empty file'

        ts = precip_time_series(input_path, temp_res=TEMP_RES)

        if ts.data['precipitation (mm/min)'].lt(0).any():
            return file, 'negative values'

        ts.pad_and_resample()

        # Check for missing timesteps
        dt_index   = ts.data.index
        full_range = pd.date_range(start=dt_index.min(), end=dt_index.max(),
                                   freq=f'{TEMP_RES}T')
        missing = full_range.difference(dt_index)
        if len(missing) > 0:
            print(f"{file}: {len(missing)} missing timesteps after resampling")

        analysis = rainfall_analysis(THRESHOLD, ts)

        if not ts.events:
            return file, 'no events found'

        # Save pickle
        with open(pickle_path, 'wb') as f:
            pickle.dump(ts, f, protocol=4)

        # Compute metrics
        analysis.get_metrics()
        df = pd.DataFrame(analysis.metrics)
        df['gauge_num']  = file.split('_')[0]
        df['start_time'] = [e[0] for e in ts.events]
        df['end_time']   = [e[1] for e in ts.events]

        df.to_csv(output_path, index=False)
        return file, f'success ({len(df)} events)'

    except Exception as e:
        return file, f'error: {e}'

# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

all_files = sorted(os.listdir(INPUT_DIR))
pending   = [f for f in all_files
             if not os.path.exists(os.path.join(OUTPUT_DIR, f'All_events_{f}'))]

print(f"Files to process: {len(pending)} / {len(all_files)}")

with Pool(processes=N_WORKERS) as pool:
    results = list(tqdm(
        pool.imap_unordered(process_file, pending),
        total=len(pending),
        desc='Processing'
    ))

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------

print("\nSummary:")
for file, status in sorted(results):
    print(f"  {file}: {status}")

errors   = [(f, s) for f, s in results if s.startswith('error')]
skipped  = [(f, s) for f, s in results if s == 'skipped']
success  = [(f, s) for f, s in results if s.startswith('success')]

print(f"\nSuccess: {len(success)}  |  Skipped: {len(skipped)}  |  Errors: {len(errors)}")


Files to process: 238 / 247


Processing:   0%|                                                                                                                                      | 0/238 [00:00<?, ?it/s]

5056
Differing duplicate rows: 134
5058
Differing duplicate rows: 131
5057
Differing duplicate rows: 120
5061
Differing duplicate rows: 131
Detected 3208 events (native mode, threshold=11h)
Detected 3217 events (native mode, threshold=11h)
Computing metrics: raw
Detected 3274 events (native mode, threshold=11h)
Computing metrics: raw
Computing metrics: raw
Computing metrics: dmc
Computing metrics: dmc
Computing metrics: dmc
Computing metrics: dblnorm
Detected 4459 events (native mode, threshold=11h)
Computing metrics: dblnorm
Computing metrics: dblnorm
Computing metrics: raw


Processing:   0%|▌                                                                                                                         | 1/238 [04:56<19:31:27, 296.57s/it]

5107
Differing duplicate rows: 60


...

Processing:   1%|█▌                                                                                                                          | 3/238 [05:03<4:29:03, 68.70s/it]

5115
Differing duplicate rows: 70


Processing:   2%|██                                                                                                                          | 4/238 [05:13<2:57:54, 45.62s/it]

5117
Differing duplicate rows: 67
Computing metrics: dmc
Detected 1606 events (native mode, threshold=11h)
Computing metrics: raw
Computing metrics: dblnorm
Computing metrics: dmc
Computing metrics: dblnorm


Processing:   2%|██                                                                                                                         | 4/238 [06:58<6:48:30, 104.75s/it]Process ForkPoolWorker-3:
Process ForkPoolWorker-2:
Process ForkPoolWorker-1:
Process ForkPoolWorker-4:


KeyboardInterrupt

